# Image Preference: воспроизводимое решение соревнования

**Задача.** Для пары сгенерированных изображений `(image_1, image_2)` предсказать вероятность того, что экспертная комиссия выбрала `image_1` как лучшее изображение. Метрика соревнования — **ROC AUC**.

**Итог.** Лучшая отправка этой ветки — `submission_soup_dino75_quality20_siglip05.csv`; она вывела команду на **2-е место** на public leaderboard с лучшим public score **0.69417**.

Ноутбук оставлен в исследовательском формате с сохраненными выводами ключевых ячеек: он показывает ход усиления baseline через новые frozen-backbone признаки и финальный ранговый ансамбль. Ячейки не перезапускались при оформлении репозитория.


## 1. Общая идея решения

Используется двухэтапный пайплайн:

1. На GPU один раз извлекаются L2-нормированные эмбеддинги изображений из замороженных `timm` backbone. Для каждого backbone сохраняются четыре `.npy`: train/test × image_1/image_2.
2. На готовых эмбеддингах обучаются быстрые pairwise-головы: Bradley-Terry MLP и LightGBM. Затем предсказания переводятся в ранговую шкалу и смешиваются.

Такой подход хорошо подходит для маленького train: тяжелые визуальные модели не дообучаются с нуля, а дают устойчивые признаки качества, композиции, семантики и артефактов генерации.


## 2. Использованные визуальные представления

В финальном решении используются четыре источника признаков:

| Backbone | Роль в ансамбле |
|---|---|
| `convnextv2_base.fcmae_ft_in22k_in1k_384` | локальные детали, текстуры, артефакты генерации |
| `vit_large_patch14_clip_224.openai` | CLIP-семантика и общий preference-сигнал |
| `vit_large_patch16_siglip_384.webli` | дополнительный vision-language сигнал на 384px |
| `vit_large_patch14_reg4_dinov2.lvd142m` | структура, композиция и self-supervised визуальные признаки |

Архив `imgpref_dinov2_work.zip` содержит уже посчитанные `.npy` для этих backbone; сам архив не добавлен в git из-за размера, но описан в `docs/ARTIFACTS.md`.


## 3. SigLIP: первая сильная прибавка к baseline

На этом шаге к исходным ConvNeXt + CLIP признакам добавляется `vit_large_patch16_siglip_384.webli`. Модель запускается по одному target-файлу в отдельном subprocess, чтобы после каждого `.npy` принудительно освобождалась RAM/VRAM Kaggle.

Результат: public score вырос с baseline примерно `0.68886` до `0.69079`.


In [5]:
from pathlib import Path
import os
import sys
import zipfile
import shutil
import subprocess
import textwrap
import json

# 1. Locate code and restore previous work/cache
runner_path = next(Path("/kaggle/input").rglob("full_train.py"))
CODE_ROOT = runner_path.parents[1]
PACKAGE_ROOT = CODE_ROOT / "image_preference"
sys.path.insert(0, str(PACKAGE_ROOT))

print("CODE_ROOT:", CODE_ROOT)

work_dir = Path("/kaggle/working/work")
cache_dir = work_dir / "cache"
work_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

work_zip_candidates = list(Path("/kaggle/input").rglob("imgpref_work_final.zip")) + list(Path("/kaggle/input").rglob("imgpref_work.zip"))
print("work zip candidates:", work_zip_candidates)

if work_zip_candidates:
    with zipfile.ZipFile(work_zip_candidates[0]) as zf:
        zf.extractall(work_dir)
    print("restored work from:", work_zip_candidates[0])
else:
    print("WARNING: no work zip found, using current /kaggle/working/work only")

print("cache after restore:")
for p in sorted(cache_dir.glob("*.npy")):
    print(" -", p.name, round(p.stat().st_size / 1024**2, 1), "MB")

# 2. Imports
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0.7"])

try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6.0"])

from imgpref.config import auto_config, BackboneSpec
from imgpref.features import cache_path
from imgpref.pipeline import run_train, run_predict

# 3. Config: old 2 backbones + new SigLIP
siglip_name = "vit_large_patch16_siglip_384.webli"

cfg = auto_config(config_path=str(PACKAGE_ROOT / "configs" / "default.yaml"))
cfg.work_dir = str(work_dir)
cfg.cache_dir = str(cache_dir)
cfg.finetune_enabled = False

# Keep old backbones exactly, append SigLIP.
cfg.backbones = list(cfg.backbones) + [
    BackboneSpec(siglip_name, img_size=384, batch_size=24),
]

cfg.decode_threads = 8
cfg.num_workers = 4
cfg.ensure_dirs()

print("backbones:")
for spec in cfg.backbones:
    print(" -", spec)

# 4. Separate-process feature extractor to hard-free RAM after each file
extract_one = Path("/kaggle/working/extract_one_feature_siglip.py")
extract_one.write_text(textwrap.dedent(r"""
    from __future__ import annotations

    import argparse
    import gc
    import os
    import sys
    from pathlib import Path

    import numpy as np

    CODE_ROOT = Path(os.environ["IMGPREF_CODE_ROOT"])
    PACKAGE_ROOT = CODE_ROOT / "image_preference"
    sys.path.insert(0, str(PACKAGE_ROOT))

    try:
        import timm
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0.7"])

    from imgpref.config import auto_config, BackboneSpec
    from imgpref.features import build_model_and_transform, cache_path, _extract_stream

    parser = argparse.ArgumentParser()
    parser.add_argument("--backbone-name", required=True)
    parser.add_argument("--img-size", type=int, required=True)
    parser.add_argument("--batch-size", type=int, required=True)
    parser.add_argument("--split", choices=["train", "test"], required=True)
    parser.add_argument("--col", choices=["image_1", "image_2"], required=True)
    args = parser.parse_args()

    cfg = auto_config(config_path=str(PACKAGE_ROOT / "configs" / "default.yaml"))
    cfg.work_dir = "/kaggle/working/work"
    cfg.cache_dir = "/kaggle/working/work/cache"
    cfg.finetune_enabled = False
    cfg.decode_threads = int(os.environ.get("IMGPREF_DECODE_THREADS", "8"))
    cfg.num_workers = int(os.environ.get("IMGPREF_NUM_WORKERS", "4"))
    cfg.ensure_dirs()

    spec = BackboneSpec(args.backbone_name, img_size=args.img_size, batch_size=args.batch_size)
    parquet = cfg.train_parquet if args.split == "train" else cfg.test_parquet
    out = cache_path(cfg, spec.name, args.split, args.col)

    if os.path.exists(out):
        print(f"[skip] cached {out}")
        raise SystemExit(0)

    print(f"[one] backbone={spec.name} split={args.split} col={args.col}")
    print(f"[one] batch_size={spec.batch_size} decode_threads={cfg.decode_threads}")

    import torch

    model, transform = build_model_and_transform(spec, cfg.pretrained, cfg.device)
    model.eval()
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)

    feats = _extract_stream(
        model=model,
        parquet=parquet,
        col=args.col,
        transform=transform,
        batch_size=spec.batch_size,
        device=cfg.device,
        amp=cfg.amp,
        tta=cfg.use_tta_hflip,
        desc=f"{spec.name.split('.')[0]} {args.split}/{args.col}",
        num_decode_threads=cfg.decode_threads,
    )

    np.save(out, feats)
    print(f"[saved] {out} shape={feats.shape}")

    del feats
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
"""), encoding="utf-8")

env = os.environ.copy()
env["IMGPREF_CODE_ROOT"] = str(CODE_ROOT)
env["IMGPREF_DECODE_THREADS"] = "8"
env["IMGPREF_NUM_WORKERS"] = "4"

# 5. Extract only SigLIP missing files
siglip_spec = cfg.backbones[-1]
for split, col in [
    ("train", cfg.img1_col),
    ("train", cfg.img2_col),
    ("test", cfg.img1_col),
    ("test", cfg.img2_col),
]:
    out = Path(cache_path(cfg, siglip_spec.name, split, col))
    if out.exists():
        print("[skip existing]", out.name)
        continue

    print("\n=== extracting SigLIP target ===")
    print(siglip_spec.name, split, col)

    subprocess.run(
        [
            sys.executable,
            str(extract_one),
            "--backbone-name", siglip_spec.name,
            "--img-size", str(siglip_spec.img_size),
            "--batch-size", str(siglip_spec.batch_size),
            "--split", split,
            "--col", col,
        ],
        check=True,
        env=env,
    )

    # checkpoint after every completed feature file
    checkpoint = shutil.make_archive(
        "/kaggle/working/imgpref_siglip_checkpoint",
        "zip",
        root_dir=str(work_dir),
    )
    print("checkpoint:", checkpoint)

print("\nAll cache files:")
for p in sorted(cache_dir.glob("*.npy")):
    print(" -", p.name, round(p.stat().st_size / 1024**2, 1), "MB")

# 6. Train and predict with 3 backbones
report = run_train(cfg)
submission = run_predict(cfg, out_path="/kaggle/working/submission_siglip.csv")

with open("/kaggle/working/report_siglip.json", "w") as f:
    json.dump(report, f, indent=2)

archive = shutil.make_archive(
    "/kaggle/working/imgpref_siglip_work",
    "zip",
    root_dir=str(work_dir),
)

print("report:", report)
print("submission:", submission)
print("archive:", archive)

CODE_ROOT: /kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1
work zip candidates: []
cache after restore:
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__test__image_1.npy 16.8 MB
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__test__image_2.npy 16.8 MB
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__train__image_1.npy 34.0 MB
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__train__image_2.npy 34.0 MB
 - feat__d3e61d7ea0_vit_large_patch14_clip_224-openai__test__image_1.npy 16.8 MB
 - feat__d3e61d7ea0_vit_large_patch14_clip_224-openai__test__image_2.npy 16.8 MB
 - feat__d3e61d7ea0_vit_large_patch14_clip_224-openai__train__image_1.npy 34.0 MB
 - feat__d3e61d7ea0_vit_large_patch14_clip_224-openai__train__image_2.npy 34.0 MB
backbones:
 - BackboneSpec(name='convnextv2_base.fcmae_ft_in22k_in1k_384', img_size=384, weight_path=None, batch_size=48)
 - BackboneSpec(name='vit_large_patch14_clip_224.openai', img_size=224, weight_path=None, 

[one] backbone=vit_large_patch16_siglip_384.webli split=train col=image_1
[one] batch_size=24 decode_threads=8


vit_large_patch16_siglip_384 train/image_1:   0%|          | 0/8710 [00:00<?, ?it/s]/kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1/image_preference/imgpref/features.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=amp):
vit_large_patch16_siglip_384 train/image_1: 100%|██████████| 8710/8710 [08:03<00:00, 18.02it/s]


[saved] /kaggle/working/work/cache/feat__192361c4ea_vit_large_patch16_siglip_384-webli__train__image_1.npy shape=(8710, 1024)
checkpoint: /kaggle/working/imgpref_siglip_checkpoint.zip

=== extracting SigLIP target ===
vit_large_patch16_siglip_384.webli train image_2
[one] backbone=vit_large_patch16_siglip_384.webli split=train col=image_2
[one] batch_size=24 decode_threads=8


vit_large_patch16_siglip_384 train/image_2:   0%|          | 0/8710 [00:00<?, ?it/s]/kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1/image_preference/imgpref/features.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=amp):
vit_large_patch16_siglip_384 train/image_2: 100%|██████████| 8710/8710 [08:30<00:00, 17.07it/s]


[saved] /kaggle/working/work/cache/feat__192361c4ea_vit_large_patch16_siglip_384-webli__train__image_2.npy shape=(8710, 1024)
checkpoint: /kaggle/working/imgpref_siglip_checkpoint.zip

=== extracting SigLIP target ===
vit_large_patch16_siglip_384.webli test image_1
[one] backbone=vit_large_patch16_siglip_384.webli split=test col=image_1
[one] batch_size=24 decode_threads=8


vit_large_patch16_siglip_384 test/image_1:   0%|          | 0/4290 [00:00<?, ?it/s]/kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1/image_preference/imgpref/features.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=amp):
vit_large_patch16_siglip_384 test/image_1: 100%|██████████| 4290/4290 [03:59<00:00, 17.92it/s]


[saved] /kaggle/working/work/cache/feat__192361c4ea_vit_large_patch16_siglip_384-webli__test__image_1.npy shape=(4290, 1024)
checkpoint: /kaggle/working/imgpref_siglip_checkpoint.zip

=== extracting SigLIP target ===
vit_large_patch16_siglip_384.webli test image_2
[one] backbone=vit_large_patch16_siglip_384.webli split=test col=image_2
[one] batch_size=24 decode_threads=8


vit_large_patch16_siglip_384 test/image_2:   0%|          | 0/4290 [00:00<?, ?it/s]/kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1/image_preference/imgpref/features.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=amp):
vit_large_patch16_siglip_384 test/image_2: 100%|██████████| 4290/4290 [04:12<00:00, 16.99it/s]


[saved] /kaggle/working/work/cache/feat__192361c4ea_vit_large_patch16_siglip_384-webli__test__image_2.npy shape=(4290, 1024)
checkpoint: /kaggle/working/imgpref_siglip_checkpoint.zip

All cache files:
 - feat__192361c4ea_vit_large_patch16_siglip_384-webli__test__image_1.npy 16.8 MB
 - feat__192361c4ea_vit_large_patch16_siglip_384-webli__test__image_2.npy 16.8 MB
 - feat__192361c4ea_vit_large_patch16_siglip_384-webli__train__image_1.npy 34.0 MB
 - feat__192361c4ea_vit_large_patch16_siglip_384-webli__train__image_2.npy 34.0 MB
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__test__image_1.npy 16.8 MB
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__test__image_2.npy 16.8 MB
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__train__image_1.npy 34.0 MB
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__train__image_2.npy 34.0 MB
 - feat__d3e61d7ea0_vit_large_patch14_clip_224-openai__test__image_1.npy 16.8 MB
 - feat__d3e61d7ea0_vit_large_patch14_

## 4. DINOv2: наиболее полезный новый backbone

Следующий эксперимент добавляет `vit_large_patch14_reg4_dinov2.lvd142m`. Локальный OOF у этой версии выглядел неоднозначно, но public leaderboard улучшился сильнее всего среди тяжелых backbone-экспериментов.

Результат: `submission_dinov2.csv` получил public score **0.69401** и стал основой финального решения.


In [6]:
from pathlib import Path
import sys, os, zipfile, shutil, subprocess, gc, json

runner = next(Path("/kaggle/input").rglob("full_train.py"))
CODE_ROOT = runner.parents[1]
PKG_ROOT = CODE_ROOT / "image_preference"
sys.path.insert(0, str(PKG_ROOT))

WORK_DIR = Path("/kaggle/working/work")
CACHE_DIR = WORK_DIR / "cache"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("CODE_ROOT:", CODE_ROOT)

# Restore cache only if current working cache is incomplete.
if len(list(CACHE_DIR.glob("feat__*.npy"))) < 12:
    names = [
        "imgpref_dinov2_work.zip",
        "imgpref_dinov2_checkpoint.zip",
        "imgpref_siglip_work.zip",
        "imgpref_siglip_checkpoint.zip",
        "imgpref_work_final.zip",
        "imgpref_work.zip",
    ]
    zips = []
    for name in names:
        zips += list(Path("/kaggle/input").rglob(name))
    if zips:
        print("restoring:", zips[0])
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORK_DIR)
    else:
        print("no work zip found; using current cache only")

from imgpref.config import auto_config, BackboneSpec
from imgpref.features import cache_path
from imgpref.pipeline import run_train, run_predict

SIGLIP = "vit_large_patch16_siglip_384.webli"
DINO = "vit_large_patch14_reg4_dinov2.lvd142m"
DINO_IMG_SIZE = 392
DINO_BATCH = 16

cfg = auto_config(config_path=str(PKG_ROOT / "configs" / "default.yaml"))
cfg.work_dir = str(WORK_DIR)
cfg.cache_dir = str(CACHE_DIR)
cfg.finetune_enabled = False
cfg.decode_threads = 8
cfg.num_workers = 2

if SIGLIP not in [s.name for s in cfg.backbones]:
    cfg.backbones.append(BackboneSpec(SIGLIP, img_size=384, batch_size=24))
if DINO not in [s.name for s in cfg.backbones]:
    cfg.backbones.append(BackboneSpec(DINO, img_size=DINO_IMG_SIZE, batch_size=DINO_BATCH))
cfg.ensure_dirs()

helper = Path("/kaggle/working/extract_one_dino.py")
helper.write_text(r'''
from pathlib import Path
import argparse, sys, gc, numpy as np

p = argparse.ArgumentParser()
p.add_argument("--code-root", required=True)
p.add_argument("--name", required=True)
p.add_argument("--img-size", type=int, required=True)
p.add_argument("--batch-size", type=int, required=True)
p.add_argument("--split", required=True)
p.add_argument("--col", required=True)
args = p.parse_args()

code_root = Path(args.code_root)
sys.path.insert(0, str(code_root / "image_preference"))

from imgpref.config import auto_config, BackboneSpec
from imgpref.features import cache_path, build_model_and_transform, _extract_stream

cfg = auto_config(config_path=str(code_root / "image_preference" / "configs" / "default.yaml"))
cfg.work_dir = "/kaggle/working/work"
cfg.cache_dir = "/kaggle/working/work/cache"
cfg.decode_threads = 8
cfg.ensure_dirs()

spec = BackboneSpec(args.name, img_size=args.img_size, batch_size=args.batch_size)
out = Path(cache_path(cfg, spec.name, args.split, args.col))
if out.exists():
    print("[skip existing]", out.name)
    raise SystemExit(0)

parquet = cfg.train_parquet if args.split == "train" else cfg.test_parquet
print(f"[one] backbone={spec.name} split={args.split} col={args.col} img={args.img_size} batch={args.batch_size}")

model, transform = build_model_and_transform(spec, cfg.pretrained, cfg.device)

import torch
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

feats = _extract_stream(
    model, parquet, args.col, transform, spec.batch_size,
    cfg.device, cfg.amp, cfg.use_tta_hflip,
    desc=f"{spec.name.split('.')[0]} {args.split}/{args.col}",
    num_decode_threads=cfg.decode_threads,
)
np.save(out, feats)
print("[saved]", out, "shape=", feats.shape)

del feats, model
gc.collect()
torch.cuda.empty_cache()
''')

for split, col in [("train", "image_1"), ("train", "image_2"), ("test", "image_1"), ("test", "image_2")]:
    out = Path(cache_path(cfg, DINO, split, col))
    if out.exists():
        print("[skip existing]", out.name)
        continue

    cmd = [
        sys.executable, str(helper),
        "--code-root", str(CODE_ROOT),
        "--name", DINO,
        "--img-size", str(DINO_IMG_SIZE),
        "--batch-size", str(DINO_BATCH),
        "--split", split,
        "--col", col,
    ]
    subprocess.run(cmd, check=True)
    gc.collect()

    ckpt = shutil.make_archive("/kaggle/working/imgpref_dinov2_checkpoint", "zip", root_dir=WORK_DIR)
    print("checkpoint:", ckpt)

print("All cache files:")
for p in sorted(CACHE_DIR.glob("feat__*.npy")):
    print(" -", p.name, round(p.stat().st_size / 1024 / 1024, 1), "MB")

report = run_train(cfg)
submission = run_predict(cfg, out_path="/kaggle/working/submission_dinov2.csv")

Path("/kaggle/working/report_dinov2.json").write_text(json.dumps(report, indent=2))
archive = shutil.make_archive("/kaggle/working/imgpref_dinov2_work", "zip", root_dir=WORK_DIR)

print("report:", report)
print("submission:", submission)
print("archive:", archive)

CODE_ROOT: /kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1


[one] backbone=vit_large_patch14_reg4_dinov2.lvd142m split=train col=image_1 img=392 batch=16


vit_large_patch14_reg4_dinov2 train/image_1:   0%|          | 0/8710 [00:00<?, ?it/s]/kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1/image_preference/imgpref/features.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=amp):
vit_large_patch14_reg4_dinov2 train/image_1: 100%|██████████| 8710/8710 [09:48<00:00, 14.79it/s]


[saved] /kaggle/working/work/cache/feat__4198903329_vit_large_patch14_reg4_dinov2-lvd142m__train__image_1.npy shape= (8710, 1024)
checkpoint: /kaggle/working/imgpref_dinov2_checkpoint.zip
[one] backbone=vit_large_patch14_reg4_dinov2.lvd142m split=train col=image_2 img=392 batch=16


vit_large_patch14_reg4_dinov2 train/image_2:   0%|          | 0/8710 [00:00<?, ?it/s]/kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1/image_preference/imgpref/features.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=amp):
vit_large_patch14_reg4_dinov2 train/image_2: 100%|██████████| 8710/8710 [10:09<00:00, 14.30it/s]


...

vit_large_patch14_reg4_dinov2 test/image_1:   0%|          | 0/4290 [00:00<?, ?it/s]/kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1/image_preference/imgpref/features.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=amp):
vit_large_patch14_reg4_dinov2 test/image_1: 100%|██████████| 4290/4290 [04:50<00:00, 14.77it/s]


[saved] /kaggle/working/work/cache/feat__4198903329_vit_large_patch14_reg4_dinov2-lvd142m__test__image_1.npy shape= (4290, 1024)
checkpoint: /kaggle/working/imgpref_dinov2_checkpoint.zip


[one] backbone=vit_large_patch14_reg4_dinov2.lvd142m split=test col=image_2 img=392 batch=16


vit_large_patch14_reg4_dinov2 test/image_2:   0%|          | 0/4290 [00:00<?, ?it/s]/kaggle/input/datasets/ivangerman/imgpref-kaggle-code-1/image_preference/imgpref/features.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=amp):
vit_large_patch14_reg4_dinov2 test/image_2: 100%|██████████| 4290/4290 [05:01<00:00, 14.21it/s]


[saved] /kaggle/working/work/cache/feat__4198903329_vit_large_patch14_reg4_dinov2-lvd142m__test__image_2.npy shape= (4290, 1024)
checkpoint: /kaggle/working/imgpref_dinov2_checkpoint.zip
All cache files:
 - feat__192361c4ea_vit_large_patch16_siglip_384-webli__test__image_1.npy 16.8 MB
 - feat__192361c4ea_vit_large_patch16_siglip_384-webli__test__image_2.npy 16.8 MB
 - feat__192361c4ea_vit_large_patch16_siglip_384-webli__train__image_1.npy 34.0 MB
 - feat__192361c4ea_vit_large_patch16_siglip_384-webli__train__image_2.npy 34.0 MB
 - feat__4198903329_vit_large_patch14_reg4_dinov2-lvd142m__test__image_1.npy 16.8 MB
 - feat__4198903329_vit_large_patch14_reg4_dinov2-lvd142m__test__image_2.npy 16.8 MB
 - feat__4198903329_vit_large_patch14_reg4_dinov2-lvd142m__train__image_1.npy 34.0 MB
 - feat__4198903329_vit_large_patch14_reg4_dinov2-lvd142m__train__image_2.npy 34.0 MB
 - feat__bb15fb641c_convnextv2_base-fcmae_ft_in22k_in1k_384__test__image_1.npy 16.8 MB
 - feat__bb15fb641c_convnextv2_base-f

## 5. Quality/artifact head

Отдельно проверялась гипотеза, что экспертный выбор зависит не только от embedding-семантики, но и от технических характеристик изображения: яркости, контраста, насыщенности, резкости, edge density, entropy, размера файла и формата.

Отдельная quality-модель была слабой, но в малой доле дала полезный независимый сигнал для итогового rank-soup.


In [7]:
from pathlib import Path
import sys, io, json, zipfile, shutil, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

runner = next(Path("/kaggle/input").rglob("full_train.py"))
CODE_ROOT = runner.parents[1]
PKG_ROOT = CODE_ROOT / "image_preference"
sys.path.insert(0, str(PKG_ROOT))

from imgpref.config import auto_config, BackboneSpec
from imgpref.pipeline import run_train
from imgpref.submit import write_submission
from imgpref.cv import search_blend_weights

WORK_DIR = Path("/kaggle/working/work")
CACHE_DIR = WORK_DIR / "cache"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not (WORK_DIR / "preds.npz").exists():
    zip_names = [
        "imgpref_dinov2_work.zip",
        "imgpref_dinov2_checkpoint.zip",
        "imgpref_siglip_work.zip",
        "imgpref_work_final.zip",
    ]
    hits = []
    for name in zip_names:
        hits += list(Path("/kaggle/input").rglob(name))
    if hits:
        print("restoring:", hits[0])
        with zipfile.ZipFile(hits[0]) as zf:
            zf.extractall(WORK_DIR)

cfg = auto_config(config_path=str(PKG_ROOT / "configs" / "default.yaml"))
cfg.work_dir = str(WORK_DIR)
cfg.cache_dir = str(CACHE_DIR)
cfg.finetune_enabled = False

for spec in [
    BackboneSpec("vit_large_patch16_siglip_384.webli", img_size=384, batch_size=24),
    BackboneSpec("vit_large_patch14_reg4_dinov2.lvd142m", img_size=392, batch_size=16),
]:
    if spec.name not in [s.name for s in cfg.backbones]:
        cfg.backbones.append(spec)

cfg.ensure_dirs()

if not (WORK_DIR / "preds.npz").exists():
    print("preds.npz not found, retraining heads from cached backbone features...")
    run_train(cfg)

RESAMPLE = getattr(Image, "Resampling", Image).BILINEAR

def image_quality_features(b):
    with Image.open(io.BytesIO(b)) as img:
        fmt = img.format or "UNK"
        w, h = img.size
        img = img.convert("RGB")
        img.thumbnail((256, 256), RESAMPLE)
        arr = np.asarray(img, dtype=np.float32) / 255.0

    r, g, bl = arr[..., 0], arr[..., 1], arr[..., 2]
    mx, mn = arr.max(axis=2), arr.min(axis=2)
    sat = (mx - mn) / (mx + 1e-6)
    gray = 0.299 * r + 0.587 * g + 0.114 * bl

    hist = np.histogram(gray, bins=32, range=(0, 1))[0].astype(np.float32)
    p = hist / (hist.sum() + 1e-8)
    entropy = float(-(p[p > 0] * np.log2(p[p > 0])).sum() / 5.0)

    if gray.shape[0] > 2 and gray.shape[1] > 2:
        lap = (
            -4 * gray[1:-1, 1:-1]
            + gray[:-2, 1:-1] + gray[2:, 1:-1]
            + gray[1:-1, :-2] + gray[1:-1, 2:]
        )
        sharp = float(lap.var())
        dx = np.abs(np.diff(gray, axis=1))
        dy = np.abs(np.diff(gray, axis=0))
        grad_mean = float((dx.mean() + dy.mean()) / 2)
        edge_density = float(((dx > 0.08).mean() + (dy > 0.08).mean()) / 2)
    else:
        sharp, grad_mean, edge_density = 0.0, 0.0, 0.0

    rg = r - g
    yb = 0.5 * (r + g) - bl
    colorfulness = float(
        np.sqrt(rg.var() + yb.var()) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2)
    )

    feats = [
        np.log1p(len(b)), np.log1p(w), np.log1p(h), np.log1p(w * h),
        w / (h + 1e-6), h / (w + 1e-6),
        float(gray.mean()), float(gray.std()), float(np.percentile(gray, 5)),
        float(np.percentile(gray, 95)), float((gray < 0.05).mean()), float((gray > 0.95).mean()),
        float(sat.mean()), float(sat.std()), float((sat < 0.05).mean()), float((sat > 0.80).mean()),
        float(r.mean()), float(g.mean()), float(bl.mean()),
        float(r.std()), float(g.std()), float(bl.std()),
        sharp, grad_mean, edge_density, entropy, colorfulness,
        float((arr < 0.02).mean()), float((arr > 0.98).mean()),
        float(fmt == "JPEG"), float(fmt == "PNG"), float(fmt == "WEBP"),
    ]
    return np.asarray(feats, dtype=np.float32)

def extract_quality(parquet_path, split, has_target):
    out_path = WORK_DIR / f"quality_{split}_v1.npz"
    if out_path.exists():
        print("cached:", out_path)
        z = np.load(out_path)
        return z["q1"], z["q2"], z["y"] if has_target else None

    cols = [cfg.img1_col, cfg.img2_col] + ([cfg.target_col] if has_target else [])
    pf = pq.ParquetFile(parquet_path)
    q1_parts, q2_parts, y_parts = [], [], []

    with ThreadPoolExecutor(max_workers=8) as ex:
        pbar = tqdm(total=pf.metadata.num_rows, desc=f"quality {split}")
        for batch in pf.iter_batches(batch_size=128, columns=cols):
            d = batch.to_pydict()
            q1_parts.append(np.stack(list(ex.map(image_quality_features, d[cfg.img1_col]))))
            q2_parts.append(np.stack(list(ex.map(image_quality_features, d[cfg.img2_col]))))
            if has_target:
                y_parts.extend(d[cfg.target_col])
            pbar.update(len(d[cfg.img1_col]))
        pbar.close()

    q1 = np.vstack(q1_parts).astype(np.float32)
    q2 = np.vstack(q2_parts).astype(np.float32)
    y = np.asarray(y_parts, dtype=np.int64) if has_target else np.array([], dtype=np.int64)
    np.savez_compressed(out_path, q1=q1, q2=q2, y=y)
    print("saved:", out_path, q1.shape, q2.shape)
    return q1, q2, y if has_target else None

def pair_quality(q1, q2):
    diff = q1 - q2
    return np.concatenate([q1, q2, diff, np.abs(diff), q1 * q2], axis=1).astype(np.float32)

q1, q2, y = extract_quality(cfg.train_parquet, "train", True)
q1t, q2t, _ = extract_quality(cfg.test_parquet, "test", False)

X = pair_quality(q1, q2)
Xt = pair_quality(q1t, q2t)
print("quality pair features:", X.shape, Xt.shape)

folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y))
q_oof = np.zeros(len(y), dtype=np.float32)
q_test = np.zeros(len(Xt), dtype=np.float32)
aucs = []

try:
    from catboost import CatBoostClassifier
    model_name = "catboost_quality"
    for fold, (tr, va) in enumerate(folds):
        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=3000,
            learning_rate=0.03,
            depth=5,
            l2_leaf_reg=10,
            random_seed=42 + fold,
            bootstrap_type="Bernoulli",
            subsample=0.85,
            od_type="Iter",
            od_wait=200,
            allow_writing_files=False,
            verbose=False,
        )
        model.fit(X[tr], y[tr], eval_set=(X[va], y[va]), use_best_model=True)
        q_oof[va] = model.predict_proba(X[va])[:, 1]
        q_test += model.predict_proba(Xt)[:, 1] / len(folds)
        auc = roc_auc_score(y[va], q_oof[va])
        aucs.append(auc)
        print(f"[quality/catboost] fold {fold}: AUC={auc:.5f}")
except Exception as e:
    print("CatBoost unavailable or failed, fallback to LightGBM:", repr(e))
    import lightgbm as lgb
    model_name = "lightgbm_quality"
    params = dict(
        objective="binary",
        metric="auc",
        learning_rate=0.025,
        num_leaves=31,
        feature_fraction=0.8,
        bagging_fraction=0.85,
        bagging_freq=1,
        min_child_samples=30,
        lambda_l2=10.0,
        verbosity=-1,
    )
    for fold, (tr, va) in enumerate(folds):
        dtr = lgb.Dataset(X[tr], label=y[tr])
        dva = lgb.Dataset(X[va], label=y[va])
        model = lgb.train(
            params, dtr, num_boost_round=3000, valid_sets=[dva],
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)],
        )
        q_oof[va] = model.predict(X[va], num_iteration=model.best_iteration)
        q_test += model.predict(Xt, num_iteration=model.best_iteration) / len(folds)
        auc = roc_auc_score(y[va], q_oof[va])
        aucs.append(auc)
        print(f"[quality/lgbm] fold {fold}: AUC={auc:.5f}")

quality_auc = roc_auc_score(y, q_oof)
print(f"[quality] OOF AUC={quality_auc:.5f}")

base = np.load(WORK_DIR / "preds.npz", allow_pickle=True)
base_oof = [base["oof"][i] for i in range(base["oof"].shape[0])]
base_test = [base["test"][i] for i in range(base["test"].shape[0])]
base_names = [str(x) for x in base["names"]]

all_oof = base_oof + [q_oof]
all_test = base_test + [q_test]
all_names = base_names + [model_name]

weights, ensemble_auc = search_blend_weights(all_oof, y, step=0.05)

def rank01(p):
    return np.argsort(np.argsort(p)).astype(np.float64) / (len(p) - 1)

ensemble_test = sum(w * rank01(p) for w, p in zip(weights, all_test))

quality_sub = write_submission(
    q_test,
    cfg.sample_submission,
    "/kaggle/working/submission_quality_only.csv",
    cfg.target_col,
)
ensemble_sub = write_submission(
    ensemble_test,
    cfg.sample_submission,
    "/kaggle/working/submission_quality_ensemble.csv",
    cfg.target_col,
)

report = {
    "quality_model": model_name,
    "quality_fold_auc": [float(x) for x in aucs],
    "quality_oof_auc": float(quality_auc),
    "ensemble_names": all_names,
    "ensemble_weights": {name: float(w) for name, w in zip(all_names, weights)},
    "ensemble_oof_auc": float(ensemble_auc),
    "quality_submission": quality_sub,
    "ensemble_submission": ensemble_sub,
}
Path("/kaggle/working/report_quality.json").write_text(json.dumps(report, indent=2))
np.savez_compressed(WORK_DIR / "quality_preds.npz", oof=q_oof, test=q_test, y=y)
archive = shutil.make_archive("/kaggle/working/imgpref_quality_work", "zip", root_dir=WORK_DIR)

print(json.dumps(report, indent=2))
print("archive:", archive)

quality train:   0%|          | 0/8710 [00:00<?, ?it/s]

saved: /kaggle/working/work/quality_train_v1.npz (8710, 32) (8710, 32)


quality test:   0%|          | 0/4290 [00:00<?, ?it/s]

saved: /kaggle/working/work/quality_test_v1.npz (4290, 32) (4290, 32)
quality pair features: (8710, 160) (4290, 160)
[quality/catboost] fold 0: AUC=0.60971
[quality/catboost] fold 1: AUC=0.61908
[quality/catboost] fold 2: AUC=0.59137
[quality/catboost] fold 3: AUC=0.62096
[quality/catboost] fold 4: AUC=0.60747
[quality] OOF AUC=0.60852
[submit] wrote /kaggle/working/submission_quality_only.csv  (4290 rows)
[submit] wrote /kaggle/working/submission_quality_ensemble.csv  (4290 rows)
{
  "quality_model": "catboost_quality",
  "quality_fold_auc": [
    0.6097080180553436,
    0.6190758629712815,
    0.5913680326819446,
    0.6209598575402413,
    0.6074660435992039
  ],
  "quality_oof_auc": 0.6085226447681839,
  "ensemble_names": [
    "bt",
    "gbdt",
    "catboost_quality"
  ],
  "ensemble_weights": {
    "bt": 0.6,
    "gbdt": 0.3,
    "catboost_quality": 0.1
  },
  "ensemble_oof_auc": 0.6654390307924727,
  "quality_submission": "/kaggle/working/submission_quality_only.csv",
  "ensembl

## 6. Финальный rank-soup

Лучшей короткой стратегией оказался консервативный ранговый ансамбль вокруг DINOv2-сабмита:

- `75%` — `submission_dinov2.csv`
- `20%` — `submission_quality_ensemble.csv`
- `5%` — `submission_siglip.csv`

Все компоненты переводятся в ранги, поэтому абсолютная калибровка вероятностей не важна; сохраняется только порядок объектов, что соответствует ROC AUC.

Именно файл `submission_soup_dino75_quality20_siglip05.csv` добавлен в `submissions/` как финальный lightweight-артефакт.


In [10]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

TARGET = "is_image1_better"

def find_csv(name):
    hits = list(Path("/kaggle/working").rglob(name)) + list(Path("/kaggle/input").rglob(name))
    if not hits:
        raise FileNotFoundError(f"Не нашел {name}")
    print(name, "->", hits[0])
    return hits[0]

def rank01(x):
    x = np.asarray(x, dtype=np.float64)
    return np.argsort(np.argsort(x)).astype(np.float64) / (len(x) - 1)

def load_ranked(name):
    path = find_csv(name)
    df = pd.read_csv(path)
    return df, rank01(df[TARGET].values)

base_df, dino = load_ranked("submission_dinov2.csv")
_, quality = load_ranked("submission_quality_ensemble.csv")
_, siglip = load_ranked("submission_siglip.csv")

blend = rank01(
    0.75 * dino +
    0.20 * quality +
    0.05 * siglip
)

out = base_df.copy()
out[TARGET] = blend

out_path = Path("/kaggle/working/submission_soup_dino75_quality20_siglip05.csv")
out.to_csv(out_path, index=False)

report = {
    "submission": str(out_path),
    "weights": {
        "dino": 0.75,
        "quality": 0.20,
        "siglip": 0.05,
    },
}
Path("/kaggle/working/report_soup_dino75_quality20_siglip05.json").write_text(json.dumps(report, indent=2))

print(json.dumps(report, indent=2))
print("wrote:", out_path)

submission_dinov2.csv -> /kaggle/working/submission_dinov2.csv
submission_quality_ensemble.csv -> /kaggle/working/submission_quality_ensemble.csv
submission_siglip.csv -> /kaggle/working/submission_siglip.csv
{
  "submission": "/kaggle/working/submission_soup_dino75_quality20_siglip05.csv",
  "weights": {
    "dino": 0.75,
    "quality": 0.2,
    "siglip": 0.05
  }
}
wrote: /kaggle/working/submission_soup_dino75_quality20_siglip05.csv


## 7. Что не вошло в финальный сабмит

Были дополнительно проверены EVA/CLIP и learned fast stack. Эти эксперименты не стали финальными:

- EVA/CLIP ухудшил public score относительно DINOv2.
- Fast stack дал интересный OOF-сигнал, но был более долгим и менее надежным для финального выбора.

Поэтому в итоговой ветке основной акцент сделан на воспроизводимом DINOv2 + SigLIP + quality rank-soup, который дал лучший public score **0.69417**.
